### **Cell 1: Imports aur State Definition**
Is cell me hum workflow ke liye zaroori libraries import kar rahe hain aur state ka structure define kar rahe hain.

* **`StateGraph, START, END`**: LangGraph ke main components hain jo graph banane aur uske starting/ending points ko define karne me kaam aate hain.
* **`TypedDict`**: Iska use pure graph me flow hone waale data ka ek fixed schema (structure) banane ke liye kiya jata hai.
* **`QuadState`**: Yeh humari custom state hai jo quadratic equation ke data (`a`, `b`, `c`, `equation`, `discriminant`, `result`) ko track karegi. Har node is state ko read aur update karega.

---

### **Cell 2: Graph Nodes (Functions) Definition**
Yahan hum un functions ko define kar rahe hain jo graph ke alag-alag **Nodes** (steps) banenge. Har function current state leta hai aur ek updated dictionary return karta hai jo state me merge ho jati hai.

* **`show_equation`**: Yeh function coefficients ($a, b, c$) ko lekar ek readable equation string banata hai.
* **`calculate_discriminant`**: Yeh quadratic formula ke discriminant ($b^2 - 4ac$) ko calculate karta hai.
* **`real_roots`**: Agar discriminant $> 0$ ho, toh yeh do alag-alag real roots calculate karta hai.
* **`repeated_roots`**: Agar discriminant $= 0$ ho, toh yeh single repeating root calculate karta hai.
* **`no_real_roots`**: Agar discriminant $< 0$ ho, toh yeh batata hai ki koi real root nahi hai (roots imaginary hain).

---

### **Cell 3: Conditional Routing Function**
Yeh function graph ko decision lene me madad karta hai.

* **`check_condition`**: Yeh ek **Routing function** hai. Yeh kisi state ko modify nahi karta, balki state ke `discriminant` ki value check karke yeh decide karta hai ki agla step (node) kaun sa hona chahiye (`real_roots`, `repeated_roots`, ya `no_real_roots`).

---

### **Cell 4: Graph Building aur Compilation**
Is cell me hum sabhi nodes aur edges ko jodkar ek complete workflow pipeline tayar kar rahe hain.



* **`graph = StateGraph(QuadState)`**: Humne `QuadState` schema ke saath ek naya graph initialize kiya.
* **`graph.add_node(...)`**: Saare functions ko graph ke andar badal diya taaki wo steps ban sakein.
* **`graph.add_edge(...)`**: Ek node se dusre node ka rasta (flow) fix kiya. Jaise: `START -> show_equation -> calculate_discriminant`.
* **`graph.add_conditional_edges(...)`**: Yahan humne routing function joda. `calculate_discriminant` ke baad kaun sa node chalega, yeh `check_condition` tai karega.
* **`workflow = graph.compile()`**: Saare nodes aur edges ko jodne ke baad graph ko compile kiya taaki hum ise execute (run) kar sakein.

---

### **Cell 5: Workflow Execution (Invocation)**
Yahan humne graph ko run kiya hai aur output check kiya hai.

* **`initial_state`**: Humne quadratic equation ke coefficients pass kiye ($a=2, b=4, c=2$). Yeh equation $2x^2 + 4x + 2 = 0$ ko represent karta hai.
* **`workflow.invoke(initial_state)`**: Isne graph ko start kiya. 
    1. Pehle equation string bani.
    2. Fir discriminant calculate hua ($4^2 - 4 \times 2 \times 2 = 0$).
    3. Conditional edge ne dekha ki discriminant `0` hai, isliye program `repeated_roots` node par gaya.
    4. Final output me hume `result: 'Only repeating root is -1.0'` mila.

In [1]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Literal

In [2]:
class QuadState(TypedDict):

    a : float
    b : float
    c : float

    equation : str
    discriminant : float
    result : str

In [3]:
def show_equation(state : QuadState):
    equation = f'{state["a"]}x^2 + {state["b"]}x + {state["c"]} = 0'

    return {'equation' : equation}


def calculate_discriminant(state : QuadState):
    discriminant = state['b']**2 - (4*state['a']*state['c'])

    return {'discriminant' : discriminant}


def real_roots(state : QuadState):

    root1 = (-state['b'] + (state['discriminant']**0.5)) / (2*state['a'])
    root2 = (state['b'] - (state['discriminant']**0.5)) / (2*state['a'])

    result = f'The roots of the equation are {root1} and {root2}'
    return {'result' : result}

def repeated_root(state : QuadState):

    root = -state['b'] / (2*state['a'])

    result = f'The equation has a repeated root: {root}'
    return {'result' : result}


def no_real_roots(state : QuadState):
    result = 'The equation has no real roots'
    return {'result' : result}

def check_condition(state : QuadState):
    if state['discriminant'] > 0:
        return "real_roots"
    elif state['discriminant'] == 0:
        return "repeated_root"
    else:
        return "no_real_roots"


In [4]:
graph = StateGraph(QuadState)

graph.add_node( 'show_equation' , show_equation)
graph.add_node( 'calculate_discriminant' , calculate_discriminant)
graph.add_node('real_roots', real_roots)
graph.add_node('repeated_root', repeated_root)
graph.add_node('no_real_roots', no_real_roots)

graph.add_edge(START , 'show_equation')
graph.add_edge('show_equation' , 'calculate_discriminant')

graph.add_conditional_edges('calculate_discriminant' , check_condition )
graph.add_edge('real_roots' , END)
graph.add_edge('repeated_root' , END)
graph.add_edge('no_real_roots' , END)

workflow = graph.compile()

In [5]:
initial_state ={
    'a' : 4 ,
    'b' : 5 ,
    'c' : 6
}
workflow.invoke(initial_state)

{'a': 4,
 'b': 5,
 'c': 6,
 'equation': '4x^2 + 5x + 6 = 0',
 'discriminant': -71,
 'result': 'The equation has no real roots'}